<a href="https://colab.research.google.com/github/bhagawatDongre/detect-open-ended-question-in-browser/blob/master/Weather_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langgraph langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 13.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.6.6
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.5
    Uninstalling langgraph-sdk-0.4.5:
      Successfully uninstalled langgraph-sdk-0.4.5
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.12
    Uninstalling langgraph-1.2.12:
      Successfully uninstalled langgraph-1.2.12


In [ ]:
import os
from google.colab import userdata

# Retrieve the secret key safely

os.environ["OPENAI_API_KEY"] =  "test_"
print("API Key loaded successfully!")

API Key loaded successfully!


In [ ]:
from typing import Literal
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# 1. Define a Mock Weather Tool
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a specific location. Use this whenever someone asks about weather conditions.

    Args:
        location: The city and state/country, e.g., 'San Francisco, CA' or 'Tokyo'
    """
    loc_lower = location.lower()
    if "london" in loc_lower:
        return "The weather in London is 15°C (59°F), Overcast with light drizzle."
    elif "new york" in loc_lower:
        return "The weather in New York is 22°C (72°F), Clear skies and sunny."
    else:
        return f"The weather in {location} is 20°C (68°F), Partly Cloudy."

# 2. Initialize Model and Bind Tools
tools = [get_weather]
tool_node = ToolNode(tools)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

# 3. Define Graph Logic
def call_model(state: MessagesState):
    messages = state['messages']
    response = model.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState) -> Literal["tools", END]:
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 4. Build and Compile the Graph Workflow
workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

app = workflow.compile()
print("Agent graph compiled and ready!")


Agent graph compiled and ready!
